## 03 - Volatilité Implicite (Smile) & Estimation GARCH(1,1)

Extraction de la volatilité implicite sur des données financières réelles via yfinance et modélisation des rendements historiques par GARCH(1,1).

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

sys.path.append(os.path.abspath('..'))
from src.volatility.implied_vol import implied_volatility
from src.volatility.garch import GARCHVolatility

# 1. Téléchargement des rendements de l'action Apple (AAPL)
df = yf.download("AAPL", start="2022-01-01", end="2026-01-01")['Close']
returns = np.log(df / df.shift(1)).dropna()

# 2. Ajustement Modèle GARCH(1,1)
garch = GARCHVolatility(returns)
res = garch.fit()

print("--- RÉSULTATS GARCH(1,1) ---")
for k, v in res.items():
    print(f"{k} : {v:.6f}")

# Volatilité annuelle prédite
vol_forecast = garch.forecast_volatility(horizon_days=1)
print(f"\nVolatilité Prédictive GARCH (1 jour) : {vol_forecast*100:.2f}%")

In [ ]:
# Construction d'un Volatility Smile synthétique
strikes = np.linspace(80, 120, 20)
S, T, r = 100.0, 0.5, 0.03

# Prix simulés du marché intégrant un smile (volatilité réelle variable)
market_prices = []
true_vols = 0.20 + 0.001 * (strikes - 100)**2  # Courbure du Smile

for K_i, vol_i in zip(strikes, true_vols):
    from src.pricing.black_scholes import bs_price
    market_prices.append(bs_price(S, K_i, T, r, vol_i, "call"))

# Reconstitution de la Volatilité Implicite via Newton-Raphson
extracted_vols = [implied_volatility(p, S, K_i, T, r, "call") for p, K_i in zip(market_prices, strikes)]

plt.figure(figsize=(8, 5))
plt.plot(strikes, extracted_vols, 'ro-', label='Volatility Smile Implicite')
plt.title("Sourire de Volatilité (Volatility Smile)")
plt.xlabel("Strike ($K$)")
plt.ylabel("Volatilité Implicite ($\sigma_{imp}$)")
plt.legend()